# 04 — Segment & Store Opportunities

Compare guest segments and rank stores for improvement pilots.

**Core logic:** `src/segment_analysis.py`, `src/opportunity_scoring.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

TABLES = ROOT / "outputs" / "tables"

## Segment differences

KPIs, top pain themes, and recommended actions by guest segment.

In [ ]:
segments = pd.read_csv(TABLES / "segment_summary.csv")
matrix = pd.read_csv(TABLES / "segment_theme_matrix.csv")
segments

In [ ]:
seg_plot = segments.copy()
seg_plot["segment_label"] = seg_plot["guest_segment"].str.replace("_", " ").str.title()

fig = px.bar(
    seg_plot.sort_values("avg_nps"),
    x="avg_nps",
    y="segment_label",
    orientation="h",
    color="avg_revisit_intent",
    title="Segment NPS (color = avg revisit intent)",
    color_continuous_scale="RdYlGn",
)
fig.show()

In [ ]:
# Segment × theme pain heatmap (negative_share from matrix)
if "negative_share" in matrix.columns:
    pivot = matrix.pivot(index="guest_segment", columns="primary_theme", values="negative_share")
    pivot.style.background_gradient(cmap="Reds", axis=None)

## Store ranking

All 90 stores ranked by composite `opportunity_score` in `store_opportunity_ranking.csv`.

In [ ]:
stores = pd.read_csv(TABLES / "store_opportunity_ranking.csv")
stores.head(10)[[
    "store_name", "region", "store_type", "nps", "negative_theme_rate",
    "opportunity_score", "top_negative_theme", "recommended_action",
]]

In [ ]:
top15 = stores.head(15).sort_values("opportunity_score")
fig = px.bar(
    top15,
    x="opportunity_score",
    y="store_name",
    color="store_type",
    orientation="h",
    title="Top 15 Stores by Opportunity Score",
)
fig.show()

## Opportunity score logic

Formula from `src/opportunity_scoring.py` (min–max indices on 0–1 scale):

```
opportunity_score =
    0.35 × negative_theme_rate_index
  + 0.25 × nps_gap_index          # worse NPS → higher index
  + 0.20 × revisit_intent_gap_index
  + 0.20 × traffic_weight         # avg_daily_transactions / portfolio max
```

Below we reproduce the index columns to show how components combine (using saved table values).

In [ ]:
recomputed = (
    0.35 * stores["negative_theme_rate_index"]
    + 0.25 * stores["nps_gap_index"]
    + 0.20 * stores["revisit_intent_gap_index"]
    + 0.20 * stores["traffic_weight"]
).round(4)

check = stores.assign(recomputed_score=recomputed)
check[["store_name", "opportunity_score", "recomputed_score"]].head(5)

In [ ]:
leader = stores.iloc[0]
print(f"#1 store: {leader['store_name']} ({leader['store_id']})")
print(f"  opportunity_score: {leader['opportunity_score']}")
print(f"  negative_theme_rate: {leader['negative_theme_rate']:.1%}")
print(f"  traffic_weight: {leader['traffic_weight']}")
print(f"  top_negative_theme: {leader['top_negative_theme']}")
print(f"  action: {leader['recommended_action']}")

## Takeaway

- **At-risk** and **price-sensitive** segments show the lowest NPS and revisit intent.
- **Mobile-first** guests over-index on `mobile_app_issues`.
- Opportunity scores concentrate in **high-traffic mall/airport** stores with elevated negative feedback.